# Reading tree-ring data with dplPy

This notebook contains a short tour of the functionality of `dplpy.readers()`. dplPy reads Tucson/ITRDB `.rwl` files (and .csv file) into a pandas `DataFrame` with one column per measurement series indexed by year.  `readers.py` has been substantially improved to be able to succeed despite the many quirks and format errors found in real ITRDB files. As of version v0.3.0 of dplPy, we have validated its behavior against dplR 1.7.9 using the whole ITRDB.

This notebook uses the sample files in `../tests/data/rwl/` in the dplPy repository.

In [11]:
# import io, os, shutil, tempfile, contextlib, warnings
import dplpy as dpl

data = dpl.readers("/Users/kja/data/itrdbMeasurementsClone/data_files/treering/measurements/northamerica/usa/wy002.rwl")
dpl.summary(data)

wy002.rwl successfully extracted as rwl file with 25 series covering the period from 1492 to 1972


,283011,283012,283021,283022,283031,283032,283041,283042,283051,283052,...,283081,283082,283091,283092,283101,283102,283111,283112,283121,283122
count,253.000000,173.000000,162.000000,160.000000,198.000000,256.000000,221.000000,197.000000,481.000000,442.000000,...,91.000000,93.000000,294.000000,264.000000,474.000000,333.000000,396.000000,393.000000,265.000000,243.000000
mean,0.949407,0.875954,1.359630,1.146188,1.553384,1.262227,0.691131,0.739036,0.529917,0.469118,...,1.296703,1.572151,0.796190,0.759962,0.548038,0.469039,0.502929,0.547786,0.987208,0.923169
std,0.594387,0.334725,0.477236,0.634681,0.414960,0.417109,0.272352,0.357044,0.280427,0.185014,...,0.356897,0.449120,0.269676,0.239188,0.323449,0.283549,0.188215,0.185544,0.328219,0.366371
min,0.080000,0.190000,0.420000,0.410000,0.670000,0.280000,0.170000,0.140000,0.060000,0.050000,...,0.480000,0.570000,0.330000,0.220000,0.060000,0.040000,0.060000,0.080000,0.340000,0.280000
25%,0.520000,0.650000,1.032500,0.747500,1.280000,1.017500,0.460000,0.480000,0.360000,0.340000,...,1.045000,1.260000,0.590000,0.587500,0.250000,0.200000,0.370000,0.420000,0.750000,0.650000
50%,0.820000,0.840000,1.230000,0.960000,1.550000,1.230000,0.730000,0.680000,0.460000,0.450000,...,1.250000,1.590000,0.740000,0.730000,0.530000,0.480000,0.500000,0.530000,0.960000,0.860000
75%,1.310000,1.100000,1.527500,1.292500,1.750000,1.510000,0.880000,0.920000,0.630000,0.580000,...,1.510000,1.890000,0.947500,0.902500,0.807500,0.670000,0.630000,0.670000,1.180000,1.165000
max,3.890000,2.170000,2.790000,4.100000,3.700000,3.250000,1.410000,2.020000,2.090000,1.200000,...,2.290000,2.620000,1.690000,1.730000,1.630000,1.600000,1.060000,1.060000,1.990000,1.980000


## 1. The basics

No arguments needed — pass a path and get back a year-indexed `DataFrame`.

In [ ]:
data = dpl.readers(DATA + "ca533.rwl")
data.iloc[:5, :5]

## 2. Headers are detected automatically

Most ITRDB files begin with a 3-line metadata header. dplPy finds where the data
actually starts — no need to pass `header=True` — and tells you how many header
lines it skipped (handy for catching a rare mis-detection).

In [ ]:
th = quiet(dpl.readers, DATA + "th001.rwl")
print("header lines skipped:", th.attrs["dplpy_header_lines_skipped"])
th.iloc[:3, :4]

## 3. Header metadata

dplPy extracts site/sample metadata from the header. It rides along on
`df.attrs["dplpy_metadata"]`, or you can pull it directly (and cheaply, reading
only the header) with `dpl.metadata()`.

In [ ]:
meta = dpl.metadata(DATA + "tx042.rwl")
meta

In [ ]:
print(meta["site_id"], "|", meta["species_code"], "-", meta["species_name"],
      "|", meta["country_region"])
print("lat/lon:", meta["latitude"], meta["longitude"],
      "| hemisphere_verified:", meta["hemisphere_verified"])

The coordinate **sign** is cross-checked against the standardized country/state
in the header (e.g. a US state forces West longitude). When the region isn't a
recognized standardized name, the sign is left as-decoded and
`hemisphere_verified` is `False`, so you know it wasn't confirmed.

## 4. Flexible about the file suffix

A Tucson file needn't end in `.rwl`. For an unrecognized suffix dplPy sniffs the
content; you can also force it with `format=`.

In [ ]:
tmp = tempfile.mkdtemp()
alt = os.path.join(tmp, "mydata.txt")            # a Tucson file with a .txt suffix
shutil.copy(DATA + "ca533.rwl", alt)
print("read a .txt by content sniffing:", quiet(dpl.readers, alt).shape)
print("or force it:", quiet(dpl.readers, alt, format="tucson").shape)

## 5. Robust to messy files — strict mode

Real archives contain malformed files. By default (`on_error="raise"`) dplPy
refuses them with a specific, actionable message rather than silently corrupting
data.

In [ ]:
for f in ["akfirmc.rwl", "viet001.rwl", "kyrg014.rwl"]:
    try:
        quiet(dpl.readers, DATA + f)
    except ValueError as e:
        detail = [ln.strip() for ln in str(e).splitlines() if ln.strip()]
        print(f"{f}:  {detail[1] if len(detail) > 1 else detail[0]}\n")

## 6. Salvage mode — warn and continue

For batch processing a whole collection, `on_error="warn"` recovers as much as
possible instead of failing: it drops an unusable series (self-overlap or a
mixed-precision series), or renames a genuinely duplicated series ID, and records
every action on `df.attrs["dplpy_salvage"]`.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d = quiet(dpl.readers, DATA + "kyrg014.rwl", on_error="warn")
print("kyrg014 salvaged ->", d.shape)
d.attrs["dplpy_salvage"]

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d2 = quiet(dpl.readers, DATA + "viet001.rwl", on_error="warn")
print("duplicate ID kept as two series:",
      [c for c in d2.columns if c.startswith("BDF02A")])
d2.attrs["dplpy_salvage"]

## 7. Tricky real-world cases

**Mixed measurement precision within one file** (here TMS* series measured at
0.001 mm and TWM* at 0.01 mm) and **non-ASCII series IDs** are both handled.

In [ ]:
tms = quiet(dpl.readers, DATA + "TMScombined.rwl")
tms[["TMS01A", "TWM01a"]].dropna().head()

In [ ]:
ru = quiet(dpl.readers, DATA + "russ301.rwl")
[c for c in ru.columns if any(ord(ch) > 127 for ch in c)][:6]

## 8. Reading straight from a URL

`readers()` accepts an http(s) URL — for example a file from the NOAA/ITRDB
archive. (This cell needs network access; it degrades gracefully if offline.)

In [ ]:
url = ("https://www.ncei.noaa.gov/pub/data/paleo/treering/"
       "measurements/northamerica/usa/ak132x.rwl")
try:
    ak = quiet(dpl.readers, url)
    print("read from URL ->", ak.shape)
except Exception as e:
    print("(network unavailable here:", type(e).__name__, "-- works on a networked machine)")

---
That's the tour: automatic headers, metadata with coordinate correction, flexible
formats and URLs, and a strict/salvage choice for handling the messy realities of
the ITRDB. See the docstring of `dpl.readers` for the full option list.